### This notebook updates the min and max values used for normalization of the risk indicator values in the BD_ClimateRisk_IKI sqlite database.

**Created:** 1/26/2025 by Sophia Bakar (sbakar@rti.org)

**Project #:** 0219481  

**Last modified:** 1/26/2025 by Sophia Bakar
 
**Status:** Complete
 
**Notes:** This script queries the IKI Climate Risk SQL database to get the max and min indicator values from the WaterALLOC indicators, Dynamic Indicators, and Static indicators, and updates the min/max values that are stored in the Indicators table. We buffer the maximum by 5% to account for significant differences in the max values of future scenarios or the addition of more basins. 

In [1]:
import pandas as pd
import sqlite3

In [2]:
# set up user and database path
#user = 'jmayo'
#user= 'cpickering'
#user = 'sgilson'
#user = 'nreynolds'
user = 'sbakar'
#db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'
db_path = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db"
# wateralloc_db = fr"C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"
wateralloc_db = fr"C:\Users\sbakar\OneDrive - Research Triangle Institute\IKI Peru Project - General\Interno\AI2b_Modelacion\Grupos_Modelacion\Resultados\BalanceHidrico.sqlite"

In [3]:
# Tables that contain indicator values
value_tables = [
    "IndValues_Dyn",
    "IndValues_Static",
    "IndValues_WaALLOC"
]

In [4]:
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

indid_sets = []

for tbl in value_tables:
    df = pd.read_sql_query(
        f"SELECT DISTINCT IndID FROM {tbl} WHERE Value IS NOT NULL;",
        conn
    )
    indid_sets.append(set(df["IndID"]))

all_indids = sorted(set.union(*indid_sets))

print(f"Found {len(all_indids)} indicators with values\n")

for ind_id in all_indids:

    ind_row = pd.read_sql_query(
        """
        SELECT Min, Max
        FROM Indicators
        WHERE IndID = ?
        """,
        conn,
        params=(ind_id,)
    )

    if ind_row.empty:
        print(f"⚠️ IndID {ind_id} not found in Indicators — skipping")
        continue

    old_min = ind_row.loc[0, "Min"]
    old_max = ind_row.loc[0, "Max"]

    values = []

    for tbl in value_tables:
        vdf = pd.read_sql_query(
            f"""
            SELECT Value
            FROM {tbl}
            WHERE IndID = ? AND Value IS NOT NULL
            """,
            conn,
            params=(ind_id,)
        )

        if not vdf.empty:
            values.extend(vdf["Value"].tolist())

    if not values:
        print(f"⚠️ IndID {ind_id}: no values found — skipping")
        continue

    raw_min = min(values)
    raw_max = max(values)

    new_min = raw_min

    # force min to be >= 0
    new_min = max(0, new_min)


    close_to_1_threshold = 1.05   # raw_max <= 1.05 → keep max=1
    expand_multiplier = 1.05      # buffer for expanded max

    if old_max == 1:
        if raw_max <= close_to_1_threshold:
            new_max = 1
        else:
            new_max = raw_max * expand_multiplier

    elif old_max == 4:
        # keep 4 fixed (same logic you had before)
        new_max = 4

    else:
        new_max = raw_max * expand_multiplier

   
    cursor.execute(
        """
        UPDATE Indicators
        SET Min = ?, Max = ?
        WHERE IndID = ?
        """,
        (new_min, new_max, ind_id)
    )

    
    print(
        f"IndID {ind_id}: "
        f"Min {old_min} → {new_min}, "
        f"Max {old_max} → {new_max} "
        f"(raw max = {raw_max})"
    )


conn.commit()
conn.close()

print("\n✅ Indicator Min/Max update complete.")

Found 52 indicators with values

IndID 101: Min 1 → 1.0, Max 4 → 4 (raw max = 4.0)
IndID 102: Min 1 → 1.0, Max 4 → 4 (raw max = 4.0)
IndID 103: Min 0.23089489200632785 → 0.23089489200632785, Max 24463.667815802095 → 24463.667815802095 (raw max = 23298.73125314485)
IndID 104: Min 1 → 1.0, Max 4 → 4 (raw max = 4.0)
IndID 105: Min 0 → 0, Max 50.2845 → 50.2845 (raw max = 47.89)
IndID 106: Min 1 → 1.0, Max 4 → 4 (raw max = 4.0)
IndID 107: Min 1 → 1.0, Max 4 → 4 (raw max = 4.0)
IndID 108: Min 0 → 0, Max 1342.0325803144951 → 1342.0325803144951 (raw max = 1278.1262669661858)
IndID 110: Min 0.25 → 0.25, Max 1 → 1 (raw max = 1.0)
IndID 111: Min 0 → 0, Max 1 → 1 (raw max = 0.75)
IndID 112: Min 0.25 → 0.25, Max 1 → 1 (raw max = 1.0)
IndID 113: Min 36.179909131955995 → 36.179909131955995, Max 5917.255359394704 → 5917.255359394704 (raw max = 5635.481294661623)
IndID 201: Min 0 → 0, Max 5184.816000000001 → 5184.816000000001 (raw max = 4937.92)
IndID 202: Min 1 → 1.0, Max 2240.7000000000003 → 2240.700